# Lab L5.2 · Production monitoring, responsible AI and champion–challenger retraining
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 5 · ⏱ 90 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Run the full MLOps maintenance cycle on two months of drifted production traffic: data quality, PSI drift, weekly performance decay, fairness and robustness checks, a retrained challenger and a promotion decision.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

It is 1 December 2025. MIZAN fraud v1.0.0 has been live for two months; chargeback labels for October and November have now arrived. The Model Risk Committee meets tomorrow and wants a **monitoring report**: is the model still fit for purpose, is it fair, is it robust, and if not, what replaces it?

**Progressive architecture:** monitors the exact package and registry you built in L5.1, and the credit policy from L3.1.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load the champion and build production features with warm-up

**Why this matters.** Production features must come from the **same module** as training and need each card’s **history** (velocity windows, ‘first time in country’). We prepend Apr–Sep history, build features, and keep the production rows. Labels (`is_fraud`) are now available for Oct–Nov.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — same module, with history warm-up

In [ ]:
import sys, json, importlib, joblib
SVC_DIR = CKPT / "mizan_fraud"                      # the deployable package lives next to the other checkpoints
sys.path.insert(0, str(CKPT))
from sklearn.metrics import average_precision_score
def at_budget(y, s, amount, budget=0.005) -> dict:
    """Ranking quality inside the fraud team's review budget (s = any score; higher = riskier)."""
    y, amount = np.asarray(y), np.asarray(amount)
    top = np.argsort(-np.asarray(s))[: int(len(y) * budget)]
    return {"precision@0.5%": y[top].mean(), "recall@0.5%": y[top].sum() / y.sum(),
            "AED_caught_share": (amount[top] * y[top]).sum() / (amount * y).sum(), "PR_AUC": average_precision_score(y, s)}
import mizan_fraud.features as F
champion = joblib.load(SVC_DIR / "model" / "fraud_rf.joblib")
card = json.loads((SVC_DIR / "model" / "model_card.json").read_text()); FEATS, THR = card["features"], card["alert_threshold"]
hist = pd.read_csv(DATA / "card_transactions.csv.gz", parse_dates=["txn_ts"])
prod = pd.read_csv(DATA / "card_transactions_prod_2025Q4.csv.gz", parse_dates=["txn_ts"])
allf = ____  # ✏️ B1
pf = allf[allf.txn_id.isin(prod.txn_id)].copy()
pf["week"] = pf.txn_ts.dt.to_period("W-SUN").dt.start_time
ref = pd.read_parquet(CKPT / "card_features.parquet").query("split == 'train'")
print(f"production rows: {len(pf):,} | weeks: {pf.week.nunique()} | champion v{card['version']} threshold {THR:.4f}")

**✔ Checkpoint.** 402k production rows over ≈ 9 weeks.

## Step 2 · Data-quality and schema monitor

**Why this matters.** Schema problems are the most common production incident and the easiest to catch. We derive **expectations from training** — allowed categories, null rates, value ranges — and test production against them with a traffic-light status.

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — any never-seen category fails the check

In [ ]:
def expectations(d: pd.DataFrame) -> dict:
    exp = {}
    for c in F.CATF:
        exp[c] = {"allowed": set(d[c].dropna().unique()), "null": d[c].isna().mean()}
    for c in F.NUMF:
        exp[c] = {"null": d[c].isna().mean(), "lo": d[c].quantile(0.001), "hi": d[c].quantile(0.999)}
    return exp

def dq_check(d: pd.DataFrame, exp: dict) -> pd.DataFrame:
    rows = []
    for c, e in exp.items():
        dn = (d[c].isna().mean() - e["null"]) * 100
        rows.append((c, "null rate change (pts)", round(dn, 2), "FAIL" if dn > 5 else "WARN" if dn > 2 else "OK"))
        if "allowed" in e:
            new = sorted(set(d[c].dropna().unique()) - e["allowed"])
            rows.append((c, "unseen categories", ", ".join(map(str, new)) or "-", ____))  # ✏️ B2
        else:
            out = ((d[c] < e["lo"]) | (d[c] > e["hi"])).mean()
            rows.append((c, "share outside ref 0.1–99.9%", round(out, 4), "WARN" if out > 0.01 else "OK"))
    return pd.DataFrame(rows, columns=["feature", "check", "observed", "status"])

dq = dq_check(pf, expectations(ref))
dq_issues = dq[dq.status != "OK"]; dq_issues

**✔ Checkpoint.** Three FAILs: unseen `Digital_Wallet` channel, unseen `UNKNOWN` merchant category, and `merchant_risk_score` null rate +7 pts (upstream enrichment outage).

## Step 3 · Data drift (PSI) and prediction drift

**Why this matters.** PSI quantifies how far each feature’s distribution has moved. **Prediction drift** — the alert rate — is visible immediately, weeks before labels. Together they are the *leading indicators*.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — PSI formula

In [ ]:
def psi(expected, actual, bins=10) -> float:
    expected, actual = pd.Series(expected).dropna(), pd.Series(actual).dropna()
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1))); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected); a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(____)  # ✏️ B3

oct_, nov_ = pf[pf.txn_ts < "2025-11-01"], pf[pf.txn_ts >= "2025-11-01"]
drift = pd.DataFrame({"PSI_Oct": [psi(ref[c], oct_[c]) for c in F.NUMF], "PSI_Nov": [psi(ref[c], nov_[c]) for c in F.NUMF]}, index=F.NUMF)
drift["status"] = np.where(drift.max(1) > 0.25, "ACT", np.where(drift.max(1) > 0.1, "WARN", "OK"))
display(drift.sort_values("PSI_Nov", ascending=False).round(3).head(10))
pf["score"] = champion.predict_proba(pf[FEATS])[:, 1]; pf["alert"] = pf.score >= THR
weekly = pf.groupby("week").agg(txns=("txn_id", "size"), alert_rate=("alert", "mean"), fraud_rate=("is_fraud", "mean"))
ax = weekly[["alert_rate", "fraud_rate"]].plot(marker="o", figsize=(10, 3.5)); ax.axhline(0.005, color="r", ls="--", label="design alert rate"); ax.legend(); plt.show()

**✔ Checkpoint.** Almost every feature is stable (PSI < 0.05). The exception is `card_txn_index` (WARN, ≈ 0.2): it counts each card’s history, so it **grows with time by construction** — a non-stationary feature that should be replaced (e.g. by transactions per month of tenure). The weekly alert rate stays near the fraud rate, so prediction drift alone would *not* raise an alarm.

## Step 4 · Performance decay and concept-drift diagnosis

**Why this matters.** Now the lagging truth. With labels we measure **recall at the operational threshold** week by week and compare it with the September test value on the model card. Then we slice by channel and 3DS status to find *where* the model fails — the signature of **concept drift**.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — slice the missed fraud by channel

In [ ]:
def weekly_perf(d):
    fr = d.is_fraud == 1
    return pd.Series({"frauds": int(fr.sum()), "recall@thr": d.loc[fr, "alert"].mean(), "precision@thr": d.loc[d.alert, "is_fraud"].mean(),
                      "PR_AUC": average_precision_score(d.is_fraud, d.score)})
perf = pf.groupby("week").apply(weekly_perf)
ref_recall = card["test_recall_at_threshold"]
ax = perf[["recall@thr", "precision@thr", "PR_AUC"]].plot(marker="o", figsize=(10, 3.8)); ax.axhline(ref_recall, color="k", ls="--", label="Sep test recall @ threshold"); ax.legend(); plt.show()
display(perf.round(3))
nov_ = pf[pf.txn_ts >= "2025-11-01"]
by_channel = nov_[nov_.is_fraud == 1].groupby(____).alert.agg(recall="mean", frauds="size")  # ✏️ B4
by_3ds = nov_[(nov_.is_fraud == 1) & (nov_.channel == "E-Commerce")].groupby("is_3ds_authenticated").alert.agg(recall="mean", frauds="size")
display(by_channel.round(3), by_3ds.round(3))

**✔ Checkpoint.** Weekly recall at the threshold ≈ 0.58–0.74 vs ≈ 0.88 on the September test — a drop well beyond the 10-point trigger. November recall on **Digital_Wallet** fraud ≈ 4% and on **3DS-authenticated** e-commerce fraud ≈ 50% (vs ≈ 85% non-3DS): the new scam typology. Classic concept drift.

## Step 5 · Responsible AI: fairness of the credit policy and robustness of the fraud model

**Why this matters.** Governance is part of maintenance. For **credit** (L3.1) we test whether approval outcomes differ by gender or nationality group — attributes the model never saw but could reach through proxies. The **disparate-impact ratio** (lowest ÷ highest approval rate; < 0.8 is a red flag) and **equal opportunity** (approval rate among borrowers who actually repaid) are standard metrics. For **fraud** we run an **evasion probe**: do fraudsters slip through if they split amounts or reuse a known device?

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — disparate-impact ratio (4/5 rule)

In [ ]:
credit = joblib.load(CKPT / "credit_model.joblib"); pol = json.loads((CKPT / "credit_policy.json").read_text())
lt = pd.read_parquet(CKPT / "loan_features.parquet").query("split == 'test'").copy()
lt["approved"] = credit.predict_proba(lt)[:, 1] < pol["pd_cutoff"]
fair = {}
for attr in ["gender", "nationality_group"]:
    t = lt.groupby(attr).apply(lambda g: pd.Series({"n": len(g), "approval_rate": g.approved.mean(),
                                                     "approval_rate_repayers": g.loc[g.default_12m == 0, "approved"].mean(),
                                                     "default_rate_if_approved": g.loc[g.approved, "default_12m"].mean()}))
    di = ____  # ✏️ B5
    fair[attr] = {"table": t, "disparate_impact": di}
    display(t.round(3)); print(f"{attr}: disparate-impact ratio = {di:.3f} → {'REVIEW' if di < 0.8 else 'within 4/5 rule'}")

In [ ]:
caught = nov_[(nov_.is_fraud == 1) & nov_.alert].copy()
probes = {}
a = caught.copy(); a["amount_aed"] /= 2; a["log_amount"] = np.log1p(a.amount_aed); a["amount_vs_card_mean"] /= 2; a["amount_sum_24h"] /= 2; a["amount_to_salary"] /= 2
probes["split amounts in half"] = (champion.predict_proba(a[FEATS])[:, 1] >= THR).mean()
b = caught.copy(); b["is_new_device"] = np.where(b.is_new_device.notna(), 0.0, np.nan)
probes["reuse a known device"] = (champion.predict_proba(b[FEATS])[:, 1] >= THR).mean()
print("share of previously-caught fraud still alerted:", {k: f"{v:.1%}" for k, v in probes.items()})

**✔ Checkpoint.** Gender DI = 1.00; nationality DI ≈ 0.94 (Emirati ≈ 95.6% vs African ≈ 90.2% approval) — inside the 4/5 rule, and default rates among approved applicants differ in the same direction (≈ 3.2% vs 4.9%), consistent with genuine risk differences via salary/employer. Evasion: ≈ 94% of caught fraud stays alerted when amounts are halved, ≈ 89% when a known device is faked — velocity features make the model fairly robust.

## Step 6 · Retrain a challenger, compare, decide, and write the report

**Why this matters.** Retraining on recent labelled data teaches the model the new scam pattern. A fair **champion–challenger** test trains the challenger on data up to October and compares both models on **November only** — data neither has seen. The decision and its evidence go to the registry and into the committee report.

#### ✏️ What to write in the next cell
- **Blank 6** *(code block, ≈2 lines)* — undersample pool: all fraud + 300k legit (random_state=RNG) → ch_train; fit a clone of the champion on it
- **Blank 7** *(inline)* — promotion rule agreed with Model Risk

In [ ]:
import os
from sklearn.base import clone
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow
from mlflow.tracking import MlflowClient
full_ref = pd.read_parquet(CKPT / "card_features.parquet")
oct_ = pf[pf.txn_ts < "2025-11-01"]; nov_ = pf[pf.txn_ts >= "2025-11-01"]
pool = pd.concat([full_ref[FEATS + ["is_fraud"]], oct_[FEATS + ["is_fraud"]]], ignore_index=True)
# ✏️ BLANK 6 — write ≈2 line(s): undersample pool: all fraud + 300k legit (random_state=RNG) → ch_train; fit a clone of the champion on it
# ...your code here...
def slice_recall(model, d, mask):
    """Recall inside the 0.5% budget, restricted to one slice of the fraud."""
    s = model.predict_proba(d[FEATS])[:, 1]; top = s >= np.quantile(s, 0.995)
    return top[(mask & (d.is_fraud == 1)).to_numpy()].mean()
cmp_rows = {}
for name, m in [("champion v1", champion), ("challenger", challenger)]:
    met = at_budget(nov_.is_fraud, m.predict_proba(nov_[FEATS])[:, 1], nov_.amount_aed)
    met["recall Digital_Wallet"] = slice_recall(m, nov_, nov_.channel == "Digital_Wallet")
    met["recall 3DS-auth e-com"] = slice_recall(m, nov_, (nov_.channel == "E-Commerce") & (nov_.is_3ds_authenticated == 1))
    cmp_rows[name] = met
comparison = pd.DataFrame(cmp_rows).T; display(comparison.round(3))
gain = comparison.loc["challenger", "PR_AUC"] - comparison.loc["champion v1", "PR_AUC"]
worse = (comparison.loc["champion v1"] - comparison.loc["challenger"]).max()
decision = "PROMOTE" if ____ else "KEEP CHAMPION"  # ✏️ B7
print(f"PR-AUC gain {gain:+.3f} | worst slice change {-worse:+.3f} → {decision}")

In [ ]:
mlflow.set_tracking_uri("sqlite:///" + (CKPT / "mlflow" / "mlflow.db").as_posix())
exp = mlflow.get_experiment_by_name("mizan-fraud"); mlflow.set_experiment(experiment_id=exp.experiment_id)
ch_dir = CKPT / "challenger_model"; ch_dir.mkdir(exist_ok=True)
joblib.dump(challenger, ch_dir / "fraud_rf.joblib", compress=3)
(ch_dir / "model_card.json").write_text(json.dumps(card | {"version": "1.1.0", "training_data": card["training_data"] + " + production Oct 2025",
    "nov_2025_eval": {k: float(v) for k, v in cmp_rows["challenger"].items()}}, indent=2))
with mlflow.start_run(run_name="challenger-retrain-oct2025") as run:
    mlflow.log_metrics({"nov_" + k.replace("@", "_at_").replace("%", "pct").replace(" ", "_").replace("-", "_"): float(v) for k, v in cmp_rows["challenger"].items()})
    mlflow.set_tag("decision", decision); mlflow.log_artifacts(str(ch_dir), artifact_path="model")
mc = MlflowClient(); mv = mc.create_model_version("mizan-fraud", f"{run.info.artifact_uri}/model", run.info.run_id)
mc.set_registered_model_alias("mizan-fraud", "champion" if decision == "PROMOTE" else "challenger", mv.version)
print(f"mizan-fraud v{mv.version} registered with alias {'champion' if decision == 'PROMOTE' else 'challenger'}")

In [ ]:
html = ["<html><head><meta charset='utf-8'><title>MIZAN fraud — monitoring report 2025 Q4</title>",
        "<style>body{font-family:Segoe UI,Arial;margin:32px;max-width:1100px}table{border-collapse:collapse;margin:8px 0 20px}td,th{border:1px solid #ccc;padding:4px 8px;font-size:13px}h2{color:#0b5e6b}</style></head><body>",
        f"<h1>MIZAN fraud — production monitoring report (Oct–Nov 2025)</h1><p>Champion v{card['version']} · generated {pd.Timestamp.now():%Y-%m-%d %H:%M} · decision: <b>{decision}</b></p>",
        "<h2>1. Data quality & schema</h2>", dq_issues.to_html(index=False),
        "<h2>2. Data drift (PSI, top 10)</h2>", drift.sort_values("PSI_Nov", ascending=False).head(10).round(3).to_html(),
        "<h2>3. Weekly performance</h2>", perf.round(3).to_html(),
        "<h2>4. Fairness — credit policy</h2>"] + [f"<h3>{a} (DI ratio {v['disparate_impact']:.3f})</h3>" + v["table"].round(3).to_html() for a, v in fair.items()] + [
        "<h2>5. Robustness probes</h2>", pd.Series(probes, name="still alerted").map("{:.1%}".format).to_frame().to_html(),
        "<h2>6. Champion vs challenger (November 2025)</h2>", comparison.round(3).to_html(), f"<p><b>Decision: {decision}</b></p></body></html>"]
(CKPT / "monitoring_report_2025Q4.html").write_text("\n".join(html), encoding="utf-8")
print("report written:", CKPT / "monitoring_report_2025Q4.html")

**✔ Checkpoint.** Challenger PR-AUC ≈ 0.76 vs 0.69 (+0.065) with no slice worse → **PROMOTE**; registry v2 takes the `champion` alias; the HTML report is written. But Digital_Wallet recall only rises from ≈ 4% to ≈ 15%: retraining helps, yet the scam typology needs **new features** (payee age, first-time beneficiary, call-centre signals) — say so in the report.

## ✅ Lab complete
You ran the full lifecycle: data → features → models → deployment → monitoring → retraining. That is the CAIP practitioner workflow end-to-end. 🎓